In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PATH DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

CHECKPOINT_DIR=os.path.join(
    BASE_DIR,
    "checkpoints"
)
print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

LOADING ALL IMAGE,AUDIO AND VIDEO EMBEDDINGS

In [ ]:
#load all image/audio and video embeddings
import numpy as np
import pandas as pd
import os

common_ids = pd.read_csv(
    f"{OUTPUT_DIR}/common_samples.csv"
)["video_id"].tolist()

image_embeddings = []
audio_embeddings = []
video_embeddings = []

for vid in common_ids:

    image_embeddings.append(
        np.load(
            os.path.join(
                IMAGE_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

    audio_embeddings.append(
        np.load(
            os.path.join(
                AUDIO_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

    video_embeddings.append(
        np.load(
            os.path.join(
                VIDEO_EMB_DIR,
                vid + ".npy"
            )
        ).squeeze()
    )

image_embeddings = np.array(image_embeddings)
audio_embeddings = np.array(audio_embeddings)
video_embeddings = np.array(video_embeddings)

DATASET PREPARATION

In [ ]:
#train/test/val split
from sklearn.model_selection import train_test_split

img_train,img_temp,\
aud_train,aud_temp,\
vid_train,vid_temp = train_test_split(
    image_embeddings,
    audio_embeddings,
    video_embeddings,
    test_size=0.30,
    random_state=42
)

img_val,img_test,\
aud_val,aud_test,\
vid_val,vid_test = train_test_split(
    img_temp,
    aud_temp,
    vid_temp,
    test_size=0.50,
    random_state=42
)

In [ ]:
#dataset
import torch
from torch.utils.data import Dataset

class FusionDataset(Dataset):

    def __init__(
        self,
        image,
        audio,
        video
    ):

        self.image = torch.tensor(
            image,
            dtype=torch.float32
        )

        self.audio = torch.tensor(
            audio,
            dtype=torch.float32
        )

        self.video = torch.tensor(
            video,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.image)

    def __getitem__(self, idx):

        return (
            self.image[idx],
            self.audio[idx],
            self.video[idx]
        )

In [ ]:
#dataloader
from torch.utils.data import DataLoader

train_loader = DataLoader(
    FusionDataset(
        img_train,
        aud_train,
        vid_train
    ),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    FusionDataset(
        img_val,
        aud_val,
        vid_val
    ),
    batch_size=32
)

test_loader = DataLoader(
    FusionDataset(
        img_test,
        aud_test,
        vid_test
    ),
    batch_size=32
)

TRANSFORMER MODEL WITH WEIGHTED LOSS

In [ ]:
#transformer model
import torch
import torch.nn as nn

class TransformerFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.image_proj = nn.Linear(
            512,
            256
        )

        self.audio_proj = nn.Linear(
            128,
            256
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.output = nn.Sequential(

            nn.Linear(
                256,
                512
            ),

            nn.ReLU(),

            nn.Linear(
                512,
                768
            )
        )

    def forward(
        self,
        image,
        audio
    ):

        image_token = self.image_proj(
            image
        )

        audio_token = self.audio_proj(
            audio
        )

        tokens = torch.stack(
            [image_token, audio_token],
            dim=1
        )

        fused = self.transformer(
            tokens
        )

        fused = fused.mean(
            dim=1
        )

        return self.output(
            fused
        )

In [ ]:
#hybrid loss
import torch.nn.functional as F

mse_loss = nn.MSELoss()

def hybrid_loss(
    pred,
    target,
    temperature=0.07,
    alpha=1.0,
    beta=1.0,
    gamma=0.5
):

    # MSE Loss
    mse = mse_loss(
        pred,
        target
    )

    # Cosine Loss
    cosine = 1 - F.cosine_similarity(
        pred,
        target,
        dim=1
    ).mean()

    # InfoNCE Loss
    pred_norm = F.normalize(
        pred,
        dim=1
    )

    target_norm = F.normalize(
        target,
        dim=1
    )

    logits = (
        pred_norm @ target_norm.T
    ) / temperature

    labels = torch.arange(
        pred.size(0)
    ).to(pred.device)

    infonce = F.cross_entropy(
        logits,
        labels
    )

    total = (
        alpha * mse
        +
        beta * cosine
        +
        gamma * infonce
    )

    return total

In [ ]:
#device gpu
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

TRAINING ON DIFFERENT GAMMA VALUES FOR INFONCE LOSS

In [ ]:
gamma_values = [0.0, 0.1, 0.3, 0.5, 1.0]

results = []

for gamma in gamma_values:

    print(f"\nRunning Gamma = {gamma}")

    model = TransformerFusion().to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4
    )

    best_val = float("inf")

    # ======================
    # TRAINING
    # ======================

    for epoch in range(20):

        model.train()

        for img,aud,vid in train_loader:

            img = img.to(device)
            aud = aud.to(device)
            vid = vid.to(device)

            optimizer.zero_grad()

            pred = model(
                img,
                aud
            )

            loss = hybrid_loss(
                pred,
                vid,
                alpha=1.0,
                beta=1.0,
                gamma=gamma
            )

            loss.backward()

            optimizer.step()

        # validation

        model.eval()

        val_loss = 0

        with torch.no_grad():

            for img,aud,vid in val_loader:

                img = img.to(device)
                aud = aud.to(device)
                vid = vid.to(device)

                pred = model(
                    img,
                    aud
                )

                loss = hybrid_loss(
                    pred,
                    vid,
                    alpha=1.0,
                    beta=1.0,
                    gamma=gamma
                )

                val_loss += loss.item()

        val_loss /= len(val_loader)

        if val_loss < best_val:

            best_val = val_loss

            torch.save(
                model.state_dict(),
                f"{CHECKPOINT_DIR}/gamma_{gamma}.pth"
            )

    # ======================
    # LOAD BEST MODEL
    # ======================

    model.load_state_dict(
        torch.load(
            f"{CHECKPOINT_DIR}/gamma_{gamma}.pth",
            map_location=device
        )
    )

    model.eval()

    # ======================
    # TEST EVALUATION
    # ======================

    all_preds = []
    all_targets = []

    with torch.no_grad():

        for img,aud,vid in test_loader:

            img = img.to(device)
            aud = aud.to(device)

            pred = model(
                img,
                aud
            )

            all_preds.append(
                pred.cpu().numpy()
            )

            all_targets.append(
                vid.numpy()
            )

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    # MSE

    mse = np.mean(
        (all_preds - all_targets) ** 2
    )

    # Cosine

    preds = normalize(all_preds)
    targets = normalize(all_targets)

    similarity_matrix = preds @ targets.T

    cosine = np.mean(
        np.diag(similarity_matrix)
    )

    # Retrieval

    ranks = []

    for i in range(len(preds)):

        sims = similarity_matrix[i]

        sorted_idx = np.argsort(
            sims
        )[::-1]

        rank = np.where(
            sorted_idx == i
        )[0][0] + 1

        ranks.append(rank)

    r1 = np.mean(
        np.array(ranks) <= 1
    ) * 100

    r5 = np.mean(
        np.array(ranks) <= 5
    ) * 100

    r10 = np.mean(
        np.array(ranks) <= 10
    ) * 100

    medr = np.median(
        ranks
    )

    results.append({

        "Gamma": gamma,

        "Cosine": round(cosine,4),

        "MSE": round(mse,4),

        "R@1": round(r1,2),

        "R@5": round(r5,2),

        "R@10": round(r10,2),

        "MedR": medr

    })

FINAL EVALUATION METRICS

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

print(results_df)

results_df.to_csv(
    f"{OUTPUT_DIR}/gamma_ablation.csv",
    index=False
)

PLOTS

In [ ]:
import matplotlib.pyplot as plt

gamma = [0.0, 0.1, 0.3, 0.5, 1.0]
r10 = [12.5, 20.0, 23.75, 38.75, 38.75]
cos = [0.9576, 0.9524, 0.9356, 0.9081, 0.8374]

fig, ax1 = plt.subplots(figsize=(8,5))

ax1.plot(gamma, r10, marker='o', label='Recall@10',color="red")
ax1.set_xlabel("Gamma")
ax1.set_ylabel("Recall@10 (%)")

ax2 = ax1.twinx()
ax2.plot(gamma, cos, marker='s', label='Cosine Similarity')
ax2.set_ylabel("Cosine Similarity")

plt.title("Effect of Gamma on Retrieval vs Reconstruction")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2)

plt.show()

In [ ]:
#parameter count
# count parameters

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
#inference latency
model.load_state_dict(
    torch.load(
        f"{CHECKPOINT_DIR}/gamma_0.5.pth",
        map_location=device
    )
)

model.eval()

In [ ]:
import time
import numpy as np

times = []

with torch.no_grad():

    for img, aud, vid in test_loader:

        img = img.to(device)
        aud = aud.to(device)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.time()

        _ = model(
            img,
            aud
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.time()

        batch_time = (
            end - start
        ) / img.size(0)

        times.append(
            batch_time
        )

avg_latency = np.mean(times) * 1000

print(
    f"Average Inference Latency: {avg_latency:.4f} ms/sample"
)